# Module 2 - Data Cleaning and Transformation

In [1]:
# Looking for the file

!hdfs dfs -ls /tmp/brazilian-ecommerce

Found 9 items
-rw-r--r--   2 root hadoop    9033957 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_customers_dataset.csv
-rw-r--r--   2 root hadoop   61273883 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_geolocation_dataset.csv
-rw-r--r--   2 root hadoop   15438671 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_items_dataset.csv
-rw-r--r--   2 root hadoop    5777138 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_payments_dataset.csv
-rw-r--r--   2 root hadoop   14451670 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_reviews_dataset.csv
-rw-r--r--   2 root hadoop   17654914 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_orders_dataset.csv
-rw-r--r--   2 root hadoop    2379446 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_products_dataset.csv
-rw-r--r--   2 root hadoop     174703 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_sellers_dataset.csv
-rw-r--r--   2 root hadoop       2613 2026-04-17 16:28 /tmp/brazilian-ecommerce/product_category_name_translation.c

In [1]:
# Creating the Spark session

from pyspark.sql import SparkSession
import spark

spark = SparkSession.builder \
        .appName("OlistData") \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 15:46:57 INFO SparkEnv: Registering MapOutputTracker
26/04/22 15:46:57 INFO SparkEnv: Registering BlockManagerMaster
26/04/22 15:46:57 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/22 15:46:57 INFO SparkEnv: Registering OutputCommitCoordinator


### Reading all the data files

In [2]:
# to simplfy I am writing thhe hdfs_path and calling the files in that folder
hdfs_path = '/tmp/brazilian-ecommerce/'

In [3]:
## Reading the customer data file
## With inferSchema as true, Spark automatically detects and assigns appropriate column data types based on the data

customers_df= spark.read.format("csv").option("header","true").option("inferSchema", "true").load(hdfs_path+'olist_customers_dataset.csv')


## Reading the olist_orders_dataset file
orders_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_orders_dataset.csv')


## Reading the olist_order_items_dataset file
order_item_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_items_dataset.csv')


## Reading the order_payments file
payments_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_payments_dataset.csv')


## Reading the olist_order_reviews_dataset file
reviews_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_reviews_dataset.csv')


## Reading the olist_products_dataset file
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema=True)


## Reading the olist_sellers_dataset file
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)


## Reading the olist_geolocation_dataset file
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)


## Reading the olist_orders_dataset file
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)

## Identifying Missing Values

In [4]:

from pyspark.sql.functions import *

def missing_values(df, df_name):
    print(f'Missing Values in {df_name}:')
    df.select([count(when(col(c).isNull(), 1)).alias(c) for c in df.columns]).show()

In [5]:
missing_values(customers_df, 'customer')

Missing Values in customer:


+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [6]:
missing_values(orders_df, 'orders')

Missing Values in orders:


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [7]:
missing_values(order_item_df, 'order_item')

Missing Values in order_item:


+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



In [8]:
missing_values(payments_df, 'payments')

Missing Values in payments:
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



In [9]:
missing_values(reviews_df, 'reviews')

Missing Values in reviews:


+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [10]:
missing_values(products_df, 'products')

Missing Values in products:
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



In [11]:
missing_values(sellers_df, 'sellers')

Missing Values in sellers:
+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



In [12]:
missing_values(geolocation_df, 'geolocation')

Missing Values in geolocation:


+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



## Handling Missing Values

1. Drop missing Values

2. Fill missing values

3. Impute Missing Values: Process of replacing missing values with estimated or default values

In [13]:
# 1. Removes rows with null/missing values
orders_df_cleaned = orders_df.na.drop( subset= ['order_id','customer_id','order_status'] ) # If only these 3 columns have null, I am dropping those rows

In [14]:
orders_df_cleaned.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [15]:
# 2. Replaces null/missing values
orders_df_cleaned = orders_df.fillna({'order_delivered_customer_date': '9999-12-31'})

In [16]:
orders_df_cleaned.show(8)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [17]:
# After the fillna; looking for the count
missing_values(orders_df_cleaned, 'orders_df_clean')

Missing Values in orders_df_clean:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                            0|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [18]:
# 3. Impute missing values
from pyspark.ml.feature import Imputer

In [19]:
#Using payments dataset: order_id, payment_sequential, payment_type, payment_installments, payment_value

payments_df.show(6) ## payment_value has no null values

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 6 rows



In [20]:
# replacing a payment_value = 99.33 with null
payments_df_with_null = payments_df.withColumn('payment_value',when(col('payment_value')!= 99.33 , col('payment_value')).otherwise(lit(None)))
payments_df_with_null.show(6)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|         NULL|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 6 rows



In [21]:
# NOW FILLING THE NULL VALUE WITH MEDIAN/MEAN


#imputer = Imputer(inputCols= ['payment_value'], outputCols= ['payment_value_imputed'] ).setStrategy('mean')

imputer = Imputer(inputCols= ['payment_value'], outputCols= ['payment_value_imputed'] ).setStrategy('median')
payments_df_cleaned = imputer.fit(payments_df_with_null).transform(payments_df_with_null)

In [22]:
payments_df_cleaned.show(6)

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|               

## Standardizing the format

In [24]:
def print_schema(df, df_name):
    print(f'Schema of {df_name}:')
    df.printSchema()
    
print_schema(orders_df, 'ORDERS')

print_schema(customers_df,'CUSTOMERS')

print_schema(payments_df,'PAYMENTS')

Schema of ORDERS:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Schema of CUSTOMERS:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

Schema of PAYMENTS:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [25]:
# Using the orders 'orders_df_cleaned' data
orders_df.show(6)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [26]:
# Converting the dates to standard formats
orders_df_cleaned = orders_df_cleaned.withColumn('order_purchase_timestamp', to_date(col('order_purchase_timestamp'))) \
                                      .withColumn('order_approved_at', to_date(col('order_approved_at'))) \
                                      .withColumn('order_delivered_carrier_date', to_date(col('order_delivered_carrier_date'))) \
                                      .withColumn('order_delivered_customer_date', to_date(col('order_delivered_customer_date'))) \
                                      .withColumn('order_estimated_delivery_date', to_date(col('order_estimated_delivery_date'))) \
        
orders_df_cleaned.show(6)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|              2017-10-02|       2017-10-02|                  2017-10-04|                   2017-10-10|                   2017-10-18|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|              2018-07-24|       2018-07-26|                  2018-07-26|                   2018-08-07|                   2018-08-13|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered

In [ ]:
# to change the column datatype

#Using Select
#orders_df_standard.select(col("order_purchase_timestamp").cast('date').alias("order_date"))

# Convert String to int
#orders_df_standard.withColumn('order_id_cast', orders_df_standard.order_id.cast('int'))

In [27]:
## payments dataset

payments_df_cleaned.select(col('payment_type')).distinct().show()
payments_df_cleaned.show(10)

+------------+
|payment_type|
+------------+
| not_defined|
| credit_card|
|      boleto|
|  debit_card|
|     voucher|
+------------+

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|     

In [28]:
payments_df_cleaned = payments_df_cleaned.withColumn('payment_type', when(col('payment_type') == 'boleto', 'Bank Transfer')
                                                                     .when(col('payment_type') == 'credit_card', 'Credit Card')
                                                                     .when(col('payment_type') == 'debit_card', 'Debit Card'))

In [29]:
payments_df_cleaned.show()

+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential| payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1|  Credit Card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1|  Credit Card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1|  Credit Card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1|  Credit Card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1|  Credit Card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1|  Credit Card|                   2|        96.12|      

## Data Type Casting

In [30]:
customers_df.printSchema() ## Need to convert customer_zip_code_prefix int to string

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [31]:
customers_df_cleaned = customers_df.withColumn('customer_zip_code_prefix', col('customer_zip_code_prefix').cast('string'))
customers_df_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



## Remove Duplicate Records

In [32]:
# dropping the duplicate records from customers_df 'customers_df_cleaned' data
customers_df_cleaned = customers_df_cleaned.dropDuplicates(['customer_id'])
customers_df_cleaned.show(3)

+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|00012a2ce6f8dcda2...|248ffe10d632bebe4...|                    6273|       osasco|            SP|
|000161a058600d590...|b0015e09bb4b6e47c...|                   35550|  itapecerica|            MG|
|000379cdec6255224...|0b83f73b19c2019e1...|                    4841|    sao paulo|            SP|
+--------------------+--------------------+------------------------+-------------+--------------+
only showing top 3 rows



## Data Transformation

In [34]:
order_with_details = orders_df_cleaned.join(order_item_df, 'order_id', 'left') \
                     .join(payments_df_cleaned, 'order_id',   'left') \
                     .join(customers_df_cleaned, 'customer_id',  'left')

In [35]:
## Find the total order value

order_with_total_value = order_with_details.groupBy('order_id').agg(sum('payment_value').alias('total_order_value'))
order_with_total_value.show(6)


+--------------------+-----------------+
|            order_id|total_order_value|
+--------------------+-----------------+
|118045506e1c1dda0...|           1802.0|
|f44cb69655f8e4d13...|           164.32|
|edcc6b79e8394346b...|           162.63|
|9f98d6530155e3b38...|           316.76|
|949280c70c6d62ec9...|            49.42|
|6a276c227b7bb9659...|           197.41|
+--------------------+-----------------+
only showing top 6 rows



In [36]:
## Delivery time calculation


delivery_detail_df = order_with_details.withColumn('delivery_time', datediff(col('order_delivered_customer_date'), col('order_purchase_timestamp'))) 

delivery_detail_df.select('order_id', 'order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_time').orderBy('delivery_time',ascending=False).show(6)

+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|2e7a8482f6fb09756...|              2016-09-04|                   9999-12-31|      2915848|
|2e7a8482f6fb09756...|              2016-09-04|                   9999-12-31|      2915848|
|e5fa5a7210941f7d5...|              2016-09-05|                   9999-12-31|      2915847|
|809a282bbd5dbcabb...|              2016-09-13|                   9999-12-31|      2915839|
|71303d7e93b399f5b...|              2016-10-02|                   9999-12-31|      2915820|
|966f96af9d189dc42...|              2016-10-04|                   9999-12-31|      2915818|
+--------------------+------------------------+-----------------------------+-------------+
only showing top 6 rows



## Advance Transformation

In [37]:
order_item_df.show(6)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
|00048cc3ae777c65d...|            1|ef92defde845

1. approxQuantile is a function used to calculate approximate percentiles of a numeric column efficiently on large datasets.
2. approxQuantile computes approximate values like median, quartiles, or percentiles without scanning all data exactly.
3. This is mainly for outlier detection
4. Values below low_cutoff → very small (possible anomalies)
5. Values above high_cutoff → very large (outliers)

In [38]:
quantiles = order_item_df.approxQuantile( 'price', [0.01,0.99], 0.0)

low_cutoff, high_cutoff = quantiles[0], quantiles[1]

In [39]:
low_cutoff, high_cutoff 

(9.99, 890.0)

In [74]:
# order_item_df.select('price').summary().show() #Not suggested on the big data

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|            112650|
|   mean|120.65373901471354|
| stddev|183.63392805026012|
|    min|              0.85|
|    25%|              39.9|
|    50%|             74.99|
|    75%|             134.9|
|    max|            6735.0|
+-------+------------------+



In [40]:
order_item_df_cleaned = order_item_df.filter((col('price') >= low_cutoff) & (col('price') <= high_cutoff))

In [41]:
# For payments_df_cleaned data

payments_df_cleaned.select('payment_installments').summary().show()

+-------+--------------------+
|summary|payment_installments|
+-------+--------------------+
|  count|              103886|
|   mean|   2.853348863176944|
| stddev|   2.687050673856492|
|    min|                   0|
|    25%|                   1|
|    50%|                   1|
|    75%|                   4|
|    max|                  24|
+-------+--------------------+



In [42]:
quantiles = payments_df_cleaned.approxQuantile( 'payment_installments', [0.05,0.99], 0.0)

low_cutoff, high_cutoff = quantiles[0], quantiles[1]

low_cutoff, high_cutoff 

(1.0, 10.0)

In [43]:
payments_df_cleaned = payments_df_cleaned.filter((col('payment_installments') >= low_cutoff) & (col('payment_installments') <= high_cutoff))
payments_df_cleaned.show(6)

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| Credit Card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1| Credit Card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| Credit Card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| Credit Card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| Credit Card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1| Credit Card|                   2|        96.12|               

In [44]:
## which product weight comes under which size(s/m/l)
products_df.show(6)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [45]:
products_df_cleaned = products_df.withColumn('product_size_category', 
                                             when(col('product_weight_g') < 500, 'Small')
                                             .when(col('product_weight_g').between(500, 2000), 'Medium')
                                             .otherwise('Large')  
                                            )

products_df_cleaned.show(8)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_size_category|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|                Small|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|        

In [46]:
## Total Revenue Per Seller: Joining the Sellers data to the order_item_df data to get the Total Revenue Per Seller

print(f'ORDER ITEM DATA: {order_item_df}')
print(f'SELLERS DATA: {sellers_df}')

ORDER ITEM DATA: DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]
SELLERS DATA: DataFrame[seller_id: string, seller_zip_code_prefix: int, seller_city: string, seller_state: string]


In [48]:
order_details_seller = order_item_df.join(sellers_df, 'seller_id', 'left')

order_details_seller.select('seller_id',  'order_id', 'order_item_id', 'price').show(6)

+--------------------+--------------------+-------------+-----+
|           seller_id|            order_id|order_item_id|price|
+--------------------+--------------------+-------------+-----+
|48436dade18ac8b2b...|00010242fe8c5a6d1...|            1| 58.9|
|dd7ddc04e1b6c2c61...|00018f77f2f0320c5...|            1|239.9|
|5b51032eddd242adc...|000229ec398224ef6...|            1|199.0|
|9d7a1d34a50524090...|00024acbcdf0a6daa...|            1|12.99|
|df560393f3a51e745...|00042b26cf59d7ce6...|            1|199.9|
|6426d21aca402a131...|00048cc3ae777c65d...|            1| 21.9|
+--------------------+--------------------+-------------+-----+
only showing top 6 rows



In [49]:
## Find the total Revenue Per Seller

order_details_seller_value = order_details_seller.groupBy('seller_id').agg(sum('price').alias('total_revenue_seller'))
order_details_seller_value.show(6)

+--------------------+--------------------+
|           seller_id|total_revenue_seller|
+--------------------+--------------------+
|8e6cc767478edae94...|   6830.580000000001|
|4d600e08ecbe08258...|             4465.34|
|9b1050e85becf3ae9...|               85.14|
|cb5ff1b9715e99589...|                85.0|
|038b75b729c8a9a04...|               467.0|
|64c9a1db4e73e19aa...|               439.0|
+--------------------+--------------------+
only showing top 6 rows



In [50]:
#!hdfs dfs -ls /tmp/brazilian-ecommerce # path of all the files 

!hadoop fs -mkdir /tmp/brazilian-ecommerce_proc # creating a folder


In [51]:
# order_with_details.write.mode('overwrite').parquet('/tmp/brazilian-ecommerce_proc/cleaned_data.parquet')
products_df_cleaned.write.mode('overwrite').parquet('/tmp/brazilian-ecommerce_proc/product_df_cleaned.parquet')

In [52]:
!hadoop fs -ls /tmp/brazilian-ecommerce_proc

Found 2 items
drwxr-xr-x   - root hadoop          0 2026-04-21 13:47 /tmp/brazilian-ecommerce_proc/cleaned_data.parquet
drwxr-xr-x   - root hadoop          0 2026-04-21 13:50 /tmp/brazilian-ecommerce_proc/product_df_cleaned.parquet


In [53]:
#order_with_details.printSchema()
products_df_cleaned.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- product_size_category: string (nullable = false)



In [ ]:
# GO to terminal -> hive -> copy paste the code -> we can access the table
CREATE EXTERNAL TABLE cleaned_payments(
    product_id STRING,
    product_category_name STRING,
    product_name_lenght STRING,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g INT,
    product_length_cm INT,
    product_height_cm INT,
    product_width_cm INT,
    product_size_category STRING
)
STORED AS PARQUET
LOCATION '/tmp/brazilian-ecommerce_proc/product_df_cleaned.parquet';